# Chapter 2 — CUDA C++ vector addition

This notebook writes the CUDA sources to files, compiles them with `nvcc`, and runs the program. It includes the supplied program and its required kernel header, with an added `--show` option to display the computed vector elements. No additional notebook extension is needed.

The program computes $c_i=a_i+b_i$ for $N=100003$ elements, copies the result back to the CPU, and validates every element. Run the cells from top to bottom.

In **Google Colab**, choose **Runtime → Change runtime type → GPU**, then connect. GPU access depends on availability. On another Jupyter installation, use a Python/IPython kernel on a Linux system with an NVIDIA GPU, compatible driver, and CUDA Toolkit.


## 1. Inspect the runtime

Colab executes the notebook on a remote Linux virtual machine. A leading `!` sends a command to that machine's shell. `nvidia-smi` reports the assigned GPU and driver information; `nvcc --version` reports the installed CUDA compiler. If either command fails, resolve the runtime setup before continuing.


In [1]:
!nvidia-smi
!nvcc --version


Sun Oct  4 07:02:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Create the source directory

The two source files will be stored together in `code/`, matching the directory layout used in the book.


In [2]:
!mkdir -p code


## 3. Write the kernel header

`%%writefile` is a built-in IPython cell command. It writes all subsequent lines of the cell to the named file, replacing that file if it exists. It does not compile or execute the C++ code.

Each thread computes one element. The `i < n` check protects array accesses in the partial final block. `extern "C"` preserves the kernel name for reuse by the Python interfaces discussed later in the book; it is not required to make the function run on the GPU.


In [3]:
%%writefile code/chapter02_vector_add_kernel.cuh
extern "C" __global__
void vector_add(const float *a, const float *b, float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}


Writing code/chapter02_vector_add_kernel.cuh


## 4. Write the complete host program

The supplied source has been extended with an optional `--show` argument for printing the first 12 results. The vector addition and validation are unchanged. It allocates GPU buffers, copies the inputs, launches the kernel, waits for completion, retrieves and validates the result, and frees the buffers. `CUDA_CHECK` reports CUDA errors.

The launch uses 256 threads per block. For 100003 elements it creates 391 blocks, so 93 threads in the last block have no corresponding array element.


In [4]:
%%writefile code/chapter02_vector_add_cuda.cu
#include <cuda_runtime.h>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <vector>
#include "chapter02_vector_add_kernel.cuh"

#define CUDA_CHECK(call) do {                                 \
    const cudaError_t status = (call);                        \
    if (status != cudaSuccess) {                              \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status));            \
        std::exit(EXIT_FAILURE);                             \
    }                                                        \
} while (0)

int main(int argc, char **argv)
{
    const bool show_results =
        argc > 1 && std::strcmp(argv[1], "--show") == 0;
    const int n = 100003;  // Not a multiple of the block size.
    const size_t bytes = size_t(n) * sizeof(float);
    std::vector<float> a(n), b(n), c(n);
    for (int i = 0; i < n; ++i) {
        a[i] = 0.25f * float(i % 101);
        b[i] = 0.50f * float(i % 37);
    }

    float *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a.data(), bytes,
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b.data(), bytes,
                          cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c.data(), d_c, bytes,
                          cudaMemcpyDeviceToHost));

    bool correct = true;
    for (int i = 0; i < n; ++i) {
        const float expected = a[i] + b[i];
        const float tolerance = 1.e-6f * (1.0f + std::fabs(expected));
        if (!std::isfinite(c[i]) ||
            std::fabs(c[i] - expected) > tolerance) {
            std::fprintf(stderr, "Mismatch at index %d\n", i);
            correct = false;
            break;
        }
    }

    if (correct && show_results) {
        const int shown = (n < 12) ? n : 12;
        std::printf("\nVector addition: first %d of %d elements\n", shown, n);
        std::puts("     i          a[i]          b[i]       c[i] (GPU)");
        for (int i = 0; i < shown; ++i) {
            std::printf("%6d  %12.6f  %12.6f  %15.6f\n",
                        i, a[i], b[i], c[i]);
        }
        std::puts("c[i] contains the actual result copied from GPU memory.");
        std::puts("Validation checks all elements, not only those displayed.");
    }

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    std::puts(correct ? "Validation passed" : "Validation failed");
    return correct ? EXIT_SUCCESS : EXIT_FAILURE;
}


Writing code/chapter02_vector_add_cuda.cu


## 5. Compile

- `nvcc` coordinates compilation of the CPU and GPU portions.
- `-O2` requests optimization level 2 for **host code**. GPU code is normally optimized independently.
- `-std=c++14` explicitly selects the C++14 language standard, which is sufficient for this example.
- `-o vector_add_cuda` names the executable.

Proceed to the execution cell only if compilation succeeds. Successful compilation may produce no output. If you change a source cell, rerun that cell and compile again before executing the program.


In [5]:
!nvcc -O2 -std=c++14 code/chapter02_vector_add_cuda.cu -o vector_add_cuda


## 6. Run and validate

The prefix `./` identifies the executable in the current directory. On success, the program prints `Validation passed`. This is the expected message, not a result already obtained in this notebook. A failed CUDA call or numerical mismatch is reported as an error.


In [6]:
!./vector_add_cuda


Validation passed


## 7. Display the computed sum

Run the following cell to display the first 12 entries of the input vectors `a`, `b`, and the result `c`. The `--show` option reruns the CUDA program and prints the actual values copied back from the GPU; it does not recompute the displayed sum in Python. All 100003 entries are still validated.

After updating the program, rerun its `%%writefile` cell and the compilation cell before using this option.


In [7]:
!./vector_add_cuda --show



Vector addition: first 12 of 100003 elements
     i          a[i]          b[i]       c[i] (GPU)
     0      0.000000      0.000000         0.000000
     1      0.250000      0.500000         0.750000
     2      0.500000      1.000000         1.500000
     3      0.750000      1.500000         2.250000
     4      1.000000      2.000000         3.000000
     5      1.250000      2.500000         3.750000
     6      1.500000      3.000000         4.500000
     7      1.750000      3.500000         5.250000
     8      2.000000      4.000000         6.000000
     9      2.250000      4.500000         6.750000
    10      2.500000      5.000000         7.500000
    11      2.750000      5.500000         8.250000
c[i] contains the actual result copied from GPU memory.
Validation checks all elements, not only those displayed.
Validation passed


## Saving and rerunning

Save the notebook in Google Drive to retain its cells. The `.cu`, `.cuh`, and executable files created in the runtime are separate from the notebook and may disappear when that virtual machine is deleted. Rerunning the writing and compilation cells recreates them. Download or explicitly save any additional result files you want to retain.

The notebook is delivered with empty outputs. Compilation and GPU execution have not been performed in the preparation environment, where `nvcc` is unavailable.
